In [1]:
import torch.nn as nn
import torch
from mpramnist.BarbadillaMartinez2026 import BarbadillaMartinezDataset
from mpramnist.BarbadillaMartinez2026 import LitModel_BarbadillaMartinez
from mpramnist.models import HumanLegNet
import mpramnist.transforms as t
from torch.utils.data import DataLoader

import lightning.pytorch as L

In [2]:
print(torch.__version__)

2.6.0+cu124


In [3]:
train_transform = t.Compose(
    [
        t.AddFlanks(BarbadillaMartinezDataset.LEFT_FLANK, BarbadillaMartinezDataset.RIGHT_FLANK),
        t.RandomPadding(600),
        t.RightCrop(600, 600),
        t.Seq2Tensor(),
    ]
)
val_transform = t.Compose(
    [
        t.AddFlanks(BarbadillaMartinezDataset.LEFT_FLANK, BarbadillaMartinezDataset.RIGHT_FLANK),
        t.Padding(600),
        t.RightCrop(600, 600),
        t.Seq2Tensor(),
    ]
)


In [4]:
train_ds = BarbadillaMartinezDataset(split=[0,1,2,3], transform=train_transform, cell_line=['AGS', 'HAP1'], root = "../data/")
val_ds = BarbadillaMartinezDataset(split=[4], transform=val_transform, cell_line=['AGS', 'HAP1'], root = "../data/")
test_ds = BarbadillaMartinezDataset(split='test', transform=val_transform, cell_line=['AGS', 'HAP1'], root = "../data/")

train_dl  = DataLoader(train_ds, batch_size=4096, num_workers=64, shuffle=True)
val_dl  = DataLoader(val_ds, batch_size=4096*2, num_workers=64, shuffle=False)


In [5]:
model = HumanLegNet(
        in_ch=4,
        output_dim=2,
        stem_ch=64,
        stem_ks=11,
        ef_ks=9,
        ef_block_sizes=[80, 96, 112, 128],
        pool_sizes=[2, 2, 2, 2],
        resize_factor=4,
    )
seq_model_AGS = LitModel_BarbadillaMartinez(
    model=model, 
    num_outputs=2,
    activity_columns=['AGS', 'HAP1'],
    #loss=nn.PoissonNLLLoss(log_input=False),
    loss=nn.MSELoss(),
    weight_decay=2e-1,
    lr=0.005, 
    print_each=1
)

In [8]:
trainer = L.Trainer(
        accelerator="gpu",
        devices=[0],
        max_epochs=1,
        gradient_clip_val=1,
        precision="16-mixed",
        enable_progress_bar=True,
        num_sanity_val_steps=0,
    )

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(seq_model_AGS, train_dataloaders=train_dl, val_dataloaders=val_dl)

In [ ]:
test_dl  = DataLoader(test_ds, batch_size=4096*2, num_workers=64, shuffle=False)
answer = trainer.predict(seq_model_AGS, dataloaders=test_dl)
a = [f['predicted'] for f in answer]
b = torch.cat(a)
b = b.numpy()
target_columns = test_ds.target_columns
cols = target_columns + ['FEAT']
real = test_ds._data[cols].copy()
pred_columns = []
for ind, ta in enumerate(target_columns):
    c = f'{ta}_pred'
    real[c] = b[:, ind]
    pred_columns.append(c)
    
ag = real.groupby('FEAT')[pred_columns + target_columns].mean()
with open(f'perf_legnet_{test_ds.version}.txt', 'a') as inp:
    for r_name, p_name in zip(target_columns, pred_columns):
        p= pearsonr(ag[r_name], ag[p_name])
        print(r_name, p)
        print(r_name, p, file=inp)
        inp.flush()